## Backstage (Lemon-Shop)

[Backstage.io](https://backstage.io) ist eine Open-Source-Developer-Plattform, die von Spotify entwickelt wurde, um die Verwaltung von Softwareprojekten zu vereinfachen. 

Mit Backstage können Unternehmen eine zentrale Anlaufstelle für all ihre internen Tools, Services, Dokumentationen und Software-Komponenten schaffen. 

Im Mittelpunkt steht das Konzept des "Service Catalogs", der alle Applikationen und Services übersichtlich darstellt. 

Durch Plugins lässt sich Backstage flexibel erweitern und an individuelle Bedürfnisse anpassen. Ziel ist es, Entwickler:innen die tägliche Arbeit zu erleichtern und eine einheitliche Nutzererfahrung über verschiedene Tools hinweg zu bieten.


### Installation

Backstage benötigt
* nvm
* node.js
* yarn (corepack)


In [ ]:
%%bash
export DEBIAN_FRONTEND=noninteractive
sudo apt-get install -y python3 g++ build-essential npm

# Download and install nvm:
curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.4/install.sh | bash
# in lieu of restarting the shell
source "$HOME/.nvm/nvm.sh"

# Download and install Node.js:
nvm install 24

# Yarn
npm install -g corepack

Wir richten jedoch unsere eigene Backstage Umgebung ein

In [ ]:
%%bash
source ~/.nvm/nvm.sh
npx @backstage/create-app@latest --path ~/lemon-shop <<<lemon-shop

### Konfiguration anpassen

Die Konfiguration kann unter folgenden URLs angepasst werden.

* [lemon-shop/app-config.yaml](../../lemon-shop/app-config.yaml)

Als erstes müssen wir den Server URL richtig setzen

In [ ]:
%%bash
sed -i -e "s:localhost:$(cat ~/data/server-ip):g" ../../lemon-shop/app-config.yaml

### Polyfill Patch

Ein **Polyfill-Patch** ergänzt fehlende Browser-Funktionen per Code, damit eine Anwendung trotzdem läuft.
In deinem Fall wird `crypto.randomUUID()` nur dann nachgerüstet, wenn der Browser diese Funktion nicht bereits bereitstellt.

Backstage braucht `crypto.randomUUID()` hier im Signals-Plugin, um eindeutige IDs für Signal-Subscriptions im Frontend zu erzeugen.


In [ ]:
%%bash
cd ~/lemon-shop || exit 1
f=packages/app/src/index.tsx

grep -q "crypto.randomUUID" "$f" && echo "Schon vorhanden" && exit 0

cp "$f" "$f.bak"
sed -i '1i\
if (globalThis.crypto && !globalThis.crypto.randomUUID) {\
  globalThis.crypto.randomUUID = (() => crypto.getRandomValues(new Uint8Array(16)).join("-")) as Crypto["randomUUID"];\
}\
' "$f"

---

### Start Backstage

Anschliessend können wir Backstage starten

In [ ]:
%%bash
source ~/data/env-platen.py
echo "http://$(cat ~/data/server-ip):3000"
source ~/.nvm/nvm.sh
cd ~/lemon-shop

yarn start

**Links**

* [Backstage.io](https://backstage.io)
* [Roadie Backstage Plug-Ins](https://roadie.io/backstage/plugins/)
* [Backstage Plug-Ins](https://backstage.io/plugins/)
* [Auto Shop Backstage](https://gitlab.com/ch-mc-b/autoshop-ms/infra/backstage)
